# CNN + interleaved diff_eml on CIFAR-10 (backprop)

The **backprop track** (the project's first gradient-trained model), and the
architecture you've been after: a **diff_eml symbolic layer *between* conv
layers**, trained end-to-end.

Why this is now possible: a hidden symbolic layer was impossible gradient-free
(no target = the credit-assignment wall). **Backprop dissolves that wall** —
gradients flow back through the diff_eml layer into the earlier conv, so the
hidden symbolic layer gets a training signal. And per the bandwidth argument,
the diff_eml layer is a **per-location symbolic channel-mixer** (a *soft
symbolic 1x1 conv*): at every spatial position the `C_in` channels become
`C_out` soft-symbolic combinations, preserving the `HxW` map for the next conv.
Many parallel symbolic units ⇒ bandwidth preserved.

`conv -> diff_eml mix -> conv -> diff_eml mix -> head`, all differentiable,
trained by backprop; the soft op/input selections anneal toward one-hot, then
we **harden** to a discrete symbolic channel-mix and **verify** the gap.

**Honest scope.** A symbolic channel-mix is a *constrained* 1x1 conv (a free
linear mix is strictly more expressive), so interleaving trades a little
accuracy for **interpretable intermediate channels** — that is the experiment.
Needs a **GPU runtime** (Runtime -> Change runtime type -> GPU).

## 1. Setup  (needs a GPU runtime)

In [ ]:
# Cell 1.1 - clone/refresh tessera, install (jax), verify GPU.
import os, sys, subprocess
REPO_DIR='tessera'; REPO_URL='https://github.com/davechendatascience/tessera.git'
def _run(c): r=subprocess.run(c,capture_output=True,text=True); return r.returncode,r.stdout,r.stderr
if not os.path.exists(REPO_DIR): print('Cloning...'); _run(['git','clone',REPO_URL])
else:
    _run(['git','-C',REPO_DIR,'fetch','--depth','1','origin','main']); _run(['git','-C',REPO_DIR,'reset','--hard','origin/main'])
_,sha,_=_run(['git','-C',REPO_DIR,'log','-1','--format=%h %s']); print('tessera @',sha.strip())
_run([sys.executable,'-m','pip','install','-e',f'./{REPO_DIR}[jax]'])
src=os.path.abspath(os.path.join(REPO_DIR,'src'))
if src not in sys.path: sys.path.insert(0,src)
import jax; print('JAX devices:',jax.devices())
if not any('cuda' in str(d).lower() or 'gpu' in str(d).lower() for d in jax.devices()):
    print('WARNING: no GPU - training will be slow. Runtime -> Change runtime type -> GPU.')

In [ ]:
# Cell 1.2 - imports.
import numpy as np, time, jax, jax.numpy as jnp
from jax import lax
from tessera.experimental.diff_sr import make_ops
tmap = jax.tree_util.tree_map
print('ready')

## 2. Load CIFAR-10

In [ ]:
# Cell 2 - load CIFAR-10 (keras/torchvision), NHWC float, contrast-normalise, subset.
def load_cifar10():
    try:
        from tensorflow.keras.datasets import cifar10
        (a,b),(c,d)=cifar10.load_data(); return a,b.ravel(),c,d.ravel()
    except Exception:
        import torchvision
        tr=torchvision.datasets.CIFAR10('./cifar',train=True,download=True)
        te=torchvision.datasets.CIFAR10('./cifar',train=False,download=True)
        return np.asarray(tr.data),np.asarray(tr.targets),np.asarray(te.data),np.asarray(te.targets)
CLASSES=['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
Xa,ya,Xb,yb=load_cifar10()
N_TRAIN,N_TEST=10000,2000        # raise N_TRAIN=50000 for the full run
rng=np.random.default_rng(0)
itr=rng.permutation(len(Xa))[:N_TRAIN]; ite=rng.permutation(len(Xb))[:N_TEST]
Xtr=(Xa[itr].astype(np.float32)/255.0); Xte=(Xb[ite].astype(np.float32)/255.0); ytr,yte=ya[itr],yb[ite]
def cn(X): m=X.mean((1,2,3),keepdims=True); s=X.std((1,2,3),keepdims=True)+1e-4; return (X-m)/s
Xtr,Xte=cn(Xtr),cn(Xte)
print('train',Xtr.shape,'test',Xte.shape)

## 3. Model: CNN with diff_eml channel-mix layers between convs

`sym_mix` is the differentiable symbolic 1x1 conv: per location, each output
channel `u` is a soft mixture over operators applied to two soft-selected input
channels — `o_u = sum_op softmax(alpha_u)[op] * op(h·softmax(bl_u), h·softmax(br_u))`.
Temperature `tau` sharpens the softmaxes toward one-hot during training.

In [ ]:
# Cell 3 - model: conv, pool, diff_eml channel-mix (near-identity init, normalised).
OPS=['add','sub','mul','div','sin','tanh','square','id']
branches,ARITIES=make_ops(OPS); K=len(OPS); ID=OPS.index('id')
def conv(x,W,b): return lax.conv_general_dilated(x,W,(1,1),'SAME',dimension_numbers=('NHWC','HWIO','NHWC'))+b
def pool(x): return lax.reduce_window(x,-jnp.inf,lax.max,(1,2,2,1),(1,2,2,1),'VALID')
def sym_mix(xmap,p,tau):                                  # (B,H,W,Cin)->(B,H,W,Cout)
    B,H,W,Cin=xmap.shape; h=xmap.reshape(-1,Cin)
    xl=h@jax.nn.softmax(p['bl']/tau,1).T; xr=h@jax.nn.softmax(p['br']/tau,1).T
    ov=jnp.clip(jnp.nan_to_num(jnp.stack([f(xl,xr) for f in branches],0)),-1e4,1e4)
    o=jnp.clip(jnp.nan_to_num(jnp.einsum('uk,kbu->bu',jax.nn.softmax(p['alpha']/tau,1),ov)),-1e4,1e4)
    o=(o-o.mean(0))/(o.std(0)+1e-3)                       # normalise (not tanh-saturate)
    return o.reshape(B,H,W,-1)
def model(p,x,tau):
    h=pool(jax.nn.relu(conv(x,p['c1W'],p['c1b']))); h=sym_mix(h,p['s1'],tau)   # conv1 -> diff_eml
    h=pool(jax.nn.relu(conv(h,p['c2W'],p['c2b']))); h=sym_mix(h,p['s2'],tau)   # conv2 -> diff_eml
    return h.reshape(h.shape[0],-1)@p['W2']+p['b2']
def loss_fn(p,x,y,tau): return -jnp.mean(jax.nn.log_softmax(model(p,x,tau))[jnp.arange(len(y)),y])
def smix(k,Ci,Co):                                       # NEAR-IDENTITY init: ch u ~ id(in u)
    bl=0.01*jax.random.normal(k,(Co,Ci)); bl=bl.at[jnp.arange(Co),jnp.arange(Co)%Ci].add(4.0)
    al=0.01*jax.random.normal(jax.random.fold_in(k,2),(Co,K)); al=al.at[:,ID].add(4.0)
    return {'bl':bl,'br':0.01*jax.random.normal(jax.random.fold_in(k,1),(Co,Ci)),'alpha':al}
def init(key,C1=32,M1=32,C2=64,M2=64):
    k=jax.random.split(key,6); he=lambda kk,sh: jax.random.normal(kk,sh)*jnp.sqrt(2.0/np.prod(np.array(sh[:-1])))
    p={'c1W':he(k[0],(3,3,3,C1)),'c1b':jnp.zeros(C1),'s1':smix(k[1],C1,M1),
       'c2W':he(k[2],(3,3,M1,C2)),'c2b':jnp.zeros(C2),'s2':smix(k[3],C2,M2)}
    h=pool(jax.nn.relu(conv(jnp.zeros((1,32,32,3)),p['c1W'],p['c1b']))); h=sym_mix(h,p['s1'],1.)
    h=pool(jax.nn.relu(conv(h,p['c2W'],p['c2b']))); h=sym_mix(h,p['s2'],1.)
    p['W2']=0.1*jax.random.normal(k[4],(h.reshape(1,-1).shape[1],10)); p['b2']=jnp.zeros(10); return p
print('model defined; ops =', OPS)

## 4. Train end-to-end (backprop; anneal tau soft -> hard)

In [ ]:
# Cell 4 - hand-rolled Adam training loop with temperature annealing.
EPOCHS, BS, LR, TAU0, TAU1 = 20, 128, 2e-3, 1.0, 0.3
@jax.jit
def train_step(p,m,v,t,xb,yb,tau):
    l,g=jax.value_and_grad(loss_fn)(p,xb,yb,tau)
    m=tmap(lambda a,b:0.9*a+0.1*b,m,g); v=tmap(lambda a,b:0.999*a+0.001*b*b,v,g)
    bc1=1-0.9**(t+1.); bc2=1-0.999**(t+1.)
    p=tmap(lambda pp,a,b: pp-LR*(a/bc1)/(jnp.sqrt(b/bc2)+1e-8),p,m,v); return p,m,v,l
def accuracy(p,X,y,tau,bs=500):
    c=0
    for s in range(0,len(X),bs):
        lo=model(p,jnp.asarray(X[s:s+bs]),jnp.float32(tau)); c+=int((np.asarray(lo).argmax(1)==y[s:s+bs]).sum())
    return c/len(X)
p=init(jax.random.PRNGKey(0)); m=tmap(jnp.zeros_like,p); v=tmap(jnp.zeros_like,p); t=0
t0=time.time()
for ep in range(EPOCHS):
    tau=float(TAU0*(TAU1/TAU0)**(ep/max(EPOCHS-1,1)))
    idx=np.random.default_rng(ep).permutation(len(Xtr))
    for s in range(0,len(Xtr),BS):
        b=idx[s:s+BS]
        p,m,v,l=train_step(p,m,v,jnp.float32(t),jnp.asarray(Xtr[b]),jnp.asarray(ytr[b]),jnp.float32(tau)); t+=1
    print(f'epoch {ep+1:2d}/{EPOCHS} tau={tau:.2f} loss={float(l):.3f} '
          f'train={accuracy(p,Xtr,ytr,tau):.3f} test={accuracy(p,Xte,yte,tau):.3f}')
print(f'trained in {time.time()-t0:.0f}s')

## 5. Harden the symbolic layers + verify

Take the argmax of each soft selection -> a **discrete** symbolic channel-mix
(`ch_u = op(in_a, in_b)`), evaluate it, and report the **discretization gap**
(soft vs hard test accuracy). Then print the discovered symbolic channel rules.

In [ ]:
# Cell 5 - harden, verify the gap, render the discovered symbolic channels.
def harden(sp): return {'op':np.asarray(sp['alpha']).argmax(1),
                        'l':np.asarray(sp['bl']).argmax(1),'r':np.asarray(sp['br']).argmax(1)}
def sym_mix_hard(xmap,h):
    B,H,W,Cin=xmap.shape; v=xmap.reshape(-1,Cin)
    cols=[branches[int(h['op'][u])](v[:,int(h['l'][u])],v[:,int(h['r'][u])]) for u in range(len(h['op']))]
    o=jnp.clip(jnp.nan_to_num(jnp.stack(cols,1)),-1e4,1e4); o=(o-o.mean(0))/(o.std(0)+1e-3)
    return o.reshape(B,H,W,-1)
def model_hard(p,x,h1,h2):
    h=pool(jax.nn.relu(conv(x,p['c1W'],p['c1b']))); h=sym_mix_hard(h,h1)
    h=pool(jax.nn.relu(conv(h,p['c2W'],p['c2b']))); h=sym_mix_hard(h,h2)
    return h.reshape(h.shape[0],-1)@p['W2']+p['b2']
h1,h2=harden(p['s1']),harden(p['s2'])
def acc_hard(X,y,bs=500):
    c=0
    for s in range(0,len(X),bs):
        c+=int((np.asarray(model_hard(p,jnp.asarray(X[s:s+bs]),h1,h2)).argmax(1)==y[s:s+bs]).sum())
    return c/len(X)
soft=accuracy(p,Xte,yte,TAU1); hard=acc_hard(Xte,yte)
print(f'TEST  soft={soft:.4f}  hardened={hard:.4f}  (discretization gap {soft-hard:+.4f})')
print('\ndiscovered symbolic channels (layer 1, first 8):')
for u in range(min(8,len(h1['op']))):
    op=int(h1['op'][u])
    if ARITIES[op]==1: print(f'  ch{u} = {OPS[op]}(in{int(h1["l"][u])})')
    else: print(f'  ch{u} = {OPS[op]}(in{int(h1["l"][u])}, in{int(h1["r"][u])})')

## What's next - honest scope

This is the **backprop track**: a CNN with **diff_eml symbolic layers between
the conv layers**, trained end-to-end, then hardened to discrete symbolic
channel-mixers. It demonstrates the thing that was impossible gradient-free —
a *hidden* symbolic layer that learns, because backprop supplies its credit.

Read the two numbers honestly: the **soft vs hardened** gap is the
discretization cost of diff_eml; and comparing this net's accuracy to a CNN
with *free* (linear) 1x1 convs in place of `sym_mix` measures what the symbolic
*constraint* costs. The payoff is interpretability — the printed `ch_u =
op(in_a, in_b)` rules are the learned symbolic channel interactions.

Levers: more filters (`C1`/`C2`/`M1`/`M2`), epochs, the op set `OPS`, a slower
tau anneal (smaller gap), full `N_TRAIN=50000`. This is **not** gradient-free
(see the csp-enhanced notebook for that track); it is the principled way to
learn conv features and symbolic structure *together*.